## Lab 4: Deploy to Production - Use AgentCore Runtime with Observability

### Overview

In Lab 3 we scaled our Customer Support Agent by centralizing tools through AgentCore Gateway with secure authentication. Now we'll complete the production journey by deploying our agent to AgentCore Runtime with comprehensive observability. This will transform our prototype into a production-ready system that can handle real-world traffic with full monitoring and automatic scaling.

[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) is a secure, fully managed runtime that empowers organizations to deploy and scale AI agents in production, regardless of framework, protocol, or model choice. It provides enterprise-grade reliability, automatic scaling, and comprehensive monitoring capabilities.

**Workshop Journey:**

- **Lab 1 (Done):** Create Agent Prototype - Built a functional customer support agent
- **Lab 2 (Done):** Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Done):** Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4 (Current):** Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5:** Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6**: Explore the User Interface - A customer-facing application

### Why AgentCore Runtime & Production Deployment Matter

Current State (Lab 1-3): Agent runs locally with centralized tools but faces production challenges:

- Agent runs locally in a single session
- No comprehensive monitoring or debugging capabilities
- Cannot handle multiple concurrent users reliably

After this lab, we will have a production-ready agent infrastructure with:

- Serverless auto-scaling to handle variable demand
- Comprehensive observability with traces, metrics, and logging
- Enterprise reliability with automatic error recovery
- Secure deployment with proper access controls
- Easy management through AWS console and APIs and support for real-world production workloads.


### Adding comprehensive observability with AgentCore Observability

Additionally, AgentCore Runtime integrates seamlessly with [AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) to provide full visibility into your agent's behavior in production. AgentCore Observability automatically captures traces, metrics, and logs from your agent interactions, tool usage, and memory access patterns. In this lab we will see how AgentCore Runtime integrates with CloudWatch GenAI Observability to provide comprehensive monitoring and debugging capabilities.

For request tracing, AgentCore Observability captures the complete conversation flow including tool invocations, memory retrievals, and model interactions. For performance monitoring, it tracks response times, success rates, and resource utilization to help optimize your agent's performance.

During the observability flow, AgentCore Runtime automatically instruments your agent code and sends telemetry data to CloudWatch. You can then use CloudWatch dashboards and GenAI Observability features to analyze patterns, identify bottlenecks, and troubleshoot issues in real-time.

### Architecture for Lab 4
<div style="text-align:left"> 
    <img src="images/architecture_lab4_runtime.png" width="75%"/> 
</div>

*Agent now runs in AgentCore Runtime with full observability through CloudWatch, serving production traffic with auto-scaling and comprehensive monitoring. Memory and Gateway integrations from previous labs remain fully functional in the production environment.*

### Key Features

- **Serverless Agent Deployment:** Transform your local agent into a scalable production service using AgentCore Runtime with minimal code changes
- **Comprehensive Observability:** Full request tracing, performance metrics, and debugging capabilities through CloudWatch GenAI Observability

### Prerequisites

- Python 3.12+
- AWS account with appropriate permissions
- Docker, Finch or Podman installed and running
- Amazon Bedrock AgentCore SDK
- Strands Agents framework
- **Infrastructure deployed:** The Gateway, Memory, and Runtime are provisioned by Terraform (`terraform apply` — see `DEPLOYMENT_GUIDE.md` at the repository root), not by running lab-03. Run [lab-03-agentcore-gateway](lab-03-agentcore-gateway.ipynb) first if you want to see/test the Gateway before testing the full Runtime here.

**Note: You MUST enable [CloudWatch Transaction Search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) to be able to see AgentCore Observability traces in CloudWatch.**


### Step 1: Import Required Libraries

In [ ]:
# Setup local AWS Credentials before all import (important!)
# Replace "iam_admin-general" by your own AWS Profile
import os

MY_AWS_PROFILE = "iam_admin-general"
os.environ["AWS_PROFILE"] = MY_AWS_PROFILE
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

In [ ]:
# Import required libraries
from lab_helpers.config import ACTOR_ID
from lab_helpers.aws_clients import REGION, agentcore_control_client
from lab_helpers.utils import get_ssm_parameter

In [ ]:
# Memory is provisioned by Terraform (modules/bedrock_agentcore_memory) — we only
# read its ID here rather than creating/fetching it via MemoryManager.
memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
print(f"Memory Id: {memory_id}")

### Step 2: How the Agent Was Adapted for AgentCore Runtime

#### From local agent to Runtime-ready agent

The production agent — [`src/customer_support_agent/main.py`](../customer_support_agent/main.py), the file actually built into the container deployed by Terraform — is exactly the Lab 1 agent with 4 small additions:

1. Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
2. Initialize the App with `app = BedrockAgentCoreApp()`
3. Decorate the invocation function with `@app.entrypoint`
4. Let AgentCore Runtime control the execution with `app.run()`

##### Key Implementation Details:

The runtime-ready agent uses an entrypoint function that extracts user prompts from the payload and JWT tokens from request headers via 
`context.request_headers.get('Authorization', '')`. The authorization token is then propagated directly to the AgentCore Gateway by passing it in the 
MCP client headers: `headers={"Authorization": auth_header}`. The implementation includes error handling for missing authentication and returns plain 
text responses from synchronous agent invocation while preserving all memory and tool functionality from previous labs.

The relevant lines in `main.py` look like this (trimmed to the Runtime-specific parts — see the file itself for the full agent logic, tools, and memory wiring):

```python
from bedrock_agentcore.runtime import BedrockAgentCoreApp  # 1. Import

app = BedrockAgentCoreApp()  # 2. Initialize

@app.entrypoint  # 3. Decorate
async def invoke(payload, context=None):
    user_input = payload.get("prompt", "")
    auth_header = (context.request_headers or {}).get("Authorization", "")
    # ... build the agent (tools, memory, gateway-backed MCP client) and invoke it ...
    response = agent(user_input)
    return response.message["content"][0]["text"]

if __name__ == "__main__":
    app.run(port=8080)  # 4. Let Runtime control execution
```

This snippet is for reference only — it's not executed by this notebook. `MEMORY_ID` and `MODEL_ID` are read from environment variables that Terraform injects into the container (`modules/bedrock_agentcore_runtime`), not from notebook globals.

#### What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

- Creates an HTTP server that listens on port 8080
- Implements the required `/invocations` endpoint for processing requests
- Implements the `/ping` endpoint for health checks
- Handles proper content types and response formats
- Manages error handling according to AWS standards


### Step 3: Connecting to the AgentCore Runtime

The AgentCore Runtime for this agent is **provisioned by Terraform** (`iac/modules/bedrock_agentcore_runtime`) — it is not created from this notebook. The container image is built/pushed and the infrastructure (Runtime, IAM role, JWT authorizer against `MCPServerPool`, request header allowlist, observability) is deployed via `terraform apply`, following the two-phase process documented in `DEPLOYMENT_GUIDE.md`.

This notebook only **connects to the already-deployed Runtime** to run test invocations.

**Note**: The Cognito access_token is valid for 2 hours only. If the access_token expires you can vend another access_token by using the `reauthenticate_user` method.

#### Reading the deployed Runtime

Terraform publishes the Runtime ARN to SSM (`/app/customersupport/agentcore/runtime_arn`) after `apply`. We read it here, along with the
data-plane endpoint, and define a small `invoke_runtime()` helper that calls the Runtime's `/invocations` HTTP API directly with a Cognito
bearer token — the same request shape the AgentCore Starter Toolkit's `Runtime.invoke()` uses internally, without needing a local
`.bedrock_agentcore.yaml` config file (which only `Runtime.configure()`/`.launch()` create).

In [ ]:
import requests
import urllib.parse
from bedrock_agentcore_starter_toolkit.utils.endpoints import get_data_plane_endpoint


# Runtime is provisioned by Terraform — read its ARN instead of configuring/launching it here.
runtime_arn = get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
runtime_id = runtime_arn.split(":")[-1].split("/")[-1]

print(f"Using existing AgentCore Runtime: {runtime_id}")
print(f"Runtime ARN: {runtime_arn}")

DP_ENDPOINT = get_data_plane_endpoint(REGION)


def invoke_runtime(payload, bearer_token, session_id, endpoint_name="DEFAULT"):
    """Invoke the Terraform-managed AgentCore Runtime via its data-plane HTTP API."""
    url = f"{DP_ENDPOINT}/runtimes/{urllib.parse.quote(runtime_arn, safe='')}/invocations"
    headers = {
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
        "Authorization": f"Bearer {bearer_token}",
    }
    
    response = requests.post(
        url,
        params={"qualifier": endpoint_name},
        headers=headers,
        json=payload,
        timeout=900
    )
    
    response.raise_for_status()
    return {"response": response.text}

#### Check Runtime Endpoint Status

Confirm the existing endpoint is `READY` (Terraform's `apply` already waits for this during deployment, so this is just a one-shot check, not a polling loop):

In [ ]:
# Setup the Agentcore Runtime Endpoint to be used
test_endpoint = "DEFAULT"

endpoint_status = agentcore_control_client.get_agent_runtime_endpoint(
    agentRuntimeId=runtime_id,
    endpointName=test_endpoint
)["status"]

print(f"Runtime endpoint status: {endpoint_status}")

In [ ]:
from lab_helpers.utils import get_or_create_cognito_pool

access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"Access token: {access_token['bearer_token']}")

### Step 4: Invoking Your Deployed Agent

Now that our agent is deployed and ready, let's test it with some queries. We invoke the agent with the right authorization token type. In our case it'll be Cognito access token. Copy the access token from the cell above

<div style="text-align:left"> 
    <img src="images/lab4_invoke.png" width="100%"/> 
</div>

#### Using `invoke_runtime()`

We use the `invoke_runtime()` helper defined above. Unlike the toolkit's `Runtime.invoke()`, it doesn't auto-generate a session id, so we create our own.

In [ ]:
# runtime_id was already read from SSM above — nothing to extract from a launch_result anymore.
print(f"Runtime ID: {runtime_id}")

In [ ]:
import uuid
from IPython.display import Markdown, display


# Create a session ID for demonstrating session continuity
session_id = uuid.uuid4()

# Test different customer support scenarios
user_query = "List all of your tools"

response = invoke_runtime(
    payload={"prompt": user_query, "actor_id": ACTOR_ID},
    bearer_token=access_token["bearer_token"],
    session_id=str(session_id),
    endpoint_name=test_endpoint,
)

display(Markdown(response["response"].replace("\\n", "\n")))

#### Invoking the agent with session continuity

Since we are using AgentCore Runtime, we can easily continue our conversation with the same `session id` and the same `Actor_id`

In [ ]:
# Test different customer support scenarios
user_query = "Tell me detailed information about the technical documentation on installing a new CPU"

response = invoke_runtime(
    payload={"prompt": user_query, "actor_id": ACTOR_ID},
    bearer_token=access_token["bearer_token"],
    session_id=str(session_id),
    endpoint_name=test_endpoint,
)

display(Markdown(response["response"].replace("\\n", "\n")))

#### Invoking the agent with a new user
In the example below our agent still has the context of the Gaming Console Pro that the user bought. This is due to the AgentCore Memory persistence. The agent won't know the context for a new user.

In [ ]:
# Creating a new session ID for demonstrating new customer
session_id2 = uuid.uuid4()

user_query = (
    "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333."
)

response = invoke_runtime(
    payload={"prompt": user_query, "actor_id": ACTOR_ID},
    bearer_token=access_token["bearer_token"],
    session_id=str(session_id2),
    endpoint_name=test_endpoint,
)
display(Markdown(response["response"].replace("\\n", "\n")))

In this case our agent does not have the context anymore and needs more information. 

And it is all it takes to have a secure and scalable endpoint for our Agent with no need to manage all the underlying infrastructure!

### Step 5: AgentCore Observability

[AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) provides monitoring and tracing capabilities for AI agents using Amazon OpenTelemetry Python Instrumentation and Amazon CloudWatch GenAI Observability.

#### Agents

Default AgentCore Runtime configuration allows for logging our agent's traces on CloudWatch by means of **AgentCore Observability**. These traces can be seen on the AWS CloudWatch GenAI Observability dashboard. Navigate to Cloudwatch &rarr; GenAI Observability &rarr; Bedrock AgentCore.

![Agents Overview on CloudWatch](images/lab4_observability_agents.png)

#### Sessions

The Sessions view shows the list of all the sessions associated with all agents in your account.

![sessions](images/lab4_observability_sessions.png)

#### Traces

Trace view lists all traces from your agents in this account. To work with traces:

- Choose Filter traces to search for specific traces.
- Sort by column name to organize results.
- Under Actions, select Logs Insights to refine your search by querying across your log and span data or select Export selected traces to export.

![traces](images/lab4_observability_traces.png)


### Congratulations! 🎉

You have successfully completed **Lab 4: Deploy to Production - Use AgentCore Runtime with Observability!**

Here is what you accomplished:

##### Production-Ready Deployment:

- Prepared your agent for production with minimal code changes (only 4 lines added)
- Validated proper session isolation between different customers
- Confirmed session continuity + memory persistence and context awareness per session

##### Enterprise-Grade Security & Identity:

- Implemented secure authentication using Cognito integration with JWT tokens
- Configured proper IAM roles and execution permissions for production workloads
- Established identity-based access control for secure agent invocation

##### Comprehensive Observability:

- Enabled AgentCore Observability for full request tracing across all customer sessions
- Configured CloudWatch GenAI Observability dashboard monitoring

##### Current Limitations (We'll fix these next!):

- **Developer Focused Interaction** - Agent accessible via SDK/API calls but no user-friendly web interface
- **Manual Session Management** - Requires programmatic session creation rather than intuitive user experience

##### Next Up [Lab 5: Evaluate Agent Performance →](lab-05-agentcore-evals.ipynb)
In Lab 5, you'll set up continuous quality monitoring with AgentCore Evaluations to ensure your production agent maintains high performance standards!
